# dataclass-training-args — worked example 2: Frozen dataclass as a hashable dict key for caching runs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclass-training-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `@dataclass(frozen=True)` makes instances immutable: assigning to a field raises `FrozenInstanceError`. Frozen dataclasses are also hashable by default (their `__hash__` is derived from the field values), so they can be used as dictionary keys or set members — handy for memoizing results keyed by an entire config.

## Worked solution

**Goal:** use a frozen `RunArgs` dataclass as a cache key so two configs with identical fields map to the same cache slot.

**Step 1 — freeze the dataclass.** `@dataclass(frozen=True)` does two things: it blocks attribute assignment after construction, and it synthesizes `__hash__` and `__eq__` from the field tuple. Both are required for use as a dict key.

**Step 2 — keep fields immutable too.** We only use scalar fields (`lr`, `batch_size`, `epochs`). A frozen dataclass with a mutable field like a `list` would still be unhashable, because hashing recurses into the list. Scalars keep the whole instance hashable.

**Step 3 — build the cache.** We make a plain dict `cache`. The key is the `RunArgs` instance itself, the value is some derived result (here a toy `lr * batch_size` cost estimate).

**Step 4 — prove value-equality.** We construct two separate `RunArgs` with the same field values. Because `__eq__` and `__hash__` come from the field tuple, `a == b` is `True` and `hash(a) == hash(b)`, so the second lookup hits the cache slot the first one created — the dict has exactly one entry even though we inserted with `a` and read with `b`.

**Why it works:** value-based hashing means “same config” collapses to “same key” automatically, which is exactly the semantics you want for caching expensive runs by their hyperparameters.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class RunArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10

cache = {}
a = RunArgs(lr=1e-3, batch_size=64)
cache[a] = a.lr * a.batch_size

b = RunArgs(lr=1e-3, batch_size=64)
print(a == b)
print(hash(a) == hash(b))
print(cache[b])
print(len(cache))